# Transfer Learning

Leveraging pre-trained models to achieve strong performance with limited data.

1. **Feature Extraction** - Freeze backbone, train only the classifier head
2. **Fine-Tuning** - Unfreeze some layers and train end-to-end with low LR
3. **When to Use Which** - Dataset size vs. similarity to pre-training data

**Task**: Classify CIFAR-10 using a pre-trained ResNet-18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ResNet expects 224x224 - resize CIFAR-10 (32x32) and use ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Use a small subset for demonstration
train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

# Subsample for speed
train_subset = torch.utils.data.Subset(train_data, range(5000))
test_subset = torch.utils.data.Subset(test_data, range(1000))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

## 1. Feature Extraction (Frozen Backbone)

In [ ]:
# Load pre-trained ResNet-18
model_fe = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in model_fe.parameters():
    param.requires_grad = False

# Replace the final fully connected layer (originally 1000 classes -> 10)
num_features = model_fe.fc.in_features
model_fe.fc = nn.Linear(num_features, 10)

model_fe = model_fe.to(device)

trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_fe.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

In [ ]:
# Train only the classifier head
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_fe.fc.parameters(), lr=1e-3)

for epoch in range(5):
    model_fe.train()
    correct, total = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model_fe(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    
    model_fe.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            test_correct += (model_fe(X_batch).argmax(1) == y_batch).sum().item()
            test_total += len(y_batch)
    
    print(f"Epoch {epoch+1} | Train: {correct/total:.4f} | Test: {test_correct/test_total:.4f}")

## 2. Fine-Tuning (Unfreezing Layers)

After feature extraction converges, unfreeze deeper layers and fine-tune with a lower learning rate.

In [ ]:
# Unfreeze the last residual block (layer4)
for param in model_fe.layer4.parameters():
    param.requires_grad = True

# Use different learning rates: lower for backbone, higher for head
optimizer_ft = optim.Adam([
    {"params": model_fe.layer4.parameters(), "lr": 1e-4},
    {"params": model_fe.fc.parameters(), "lr": 1e-3},
])

trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f"Now trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

for epoch in range(5):
    model_fe.train()
    correct, total_n = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_ft.zero_grad()
        logits = model_fe(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer_ft.step()
        correct += (logits.argmax(1) == y_batch).sum().item()
        total_n += len(y_batch)
    
    model_fe.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            test_correct += (model_fe(X_batch).argmax(1) == y_batch).sum().item()
            test_total += len(y_batch)
    
    print(f"Fine-tune Epoch {epoch+1} | Train: {correct/total_n:.4f} | Test: {test_correct/test_total:.4f}")

## Transfer Learning Strategy Guide

| Data Size | Similar to Pre-training | Different from Pre-training |
|-----------|------------------------|---------------------------|
| **Small** | Feature extraction (freeze all) | Feature extraction + augmentation |
| **Large** | Fine-tune top layers | Fine-tune all layers with low LR |

## Key Takeaways

1. **Pre-trained models encode universal features** - edges, textures, shapes in early layers
2. **Feature extraction first, then fine-tune** - this two-stage approach is more stable
3. **Use lower LR for pre-trained layers** - avoid destroying learned representations
4. **Data augmentation is even more critical** with small datasets
5. **Transfer learning works across domains** - ImageNet features transfer to medical images, satellite imagery, etc.